In [96]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [97]:
data=pd.read_csv('all-data.csv',encoding='latin-1',header=None)
data.head()

,0,1
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...


In [98]:
print(data.shape)

(4846, 2)


In [99]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4846 entries, 0 to 4845
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   0       4846 non-null   str  
 1   1       4846 non-null   str  
dtypes: str(2)
memory usage: 75.8 KB


In [100]:
print(data.columns)

Index([0, 1], dtype='int64')


In [101]:
data.columns=['sentimental','news']

In [102]:
data.head()

,sentimental,news
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...


In [103]:
print(data.columns)

Index(['sentimental', 'news'], dtype='str')


In [104]:
print(data['sentimental'].value_counts())

sentimental
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64


In [105]:
print(data.isnull().sum())

sentimental    0
news           0
dtype: int64


In [106]:
print(data.duplicated().sum() )

6


In [107]:
data.drop_duplicates(inplace=True)

In [108]:
print(data.duplicated().sum() )

0


In [109]:
data['length']=data['news'].apply(len)
print(data['length'].describe())

count    4840.000000
mean      128.076446
std        56.416479
min         9.000000
25%        84.000000
50%       119.000000
75%       163.000000
max       315.000000
Name: length, dtype: float64


In [110]:
data.head()

,sentimental,news,length
0,neutral,"According to Gran , the company has no plans t...",127
1,neutral,Technopolis plans to develop in stages an area...,190
2,negative,The international electronic industry company ...,228
3,positive,With the new production plant the company woul...,206
4,positive,According to the company 's updated strategy f...,203


In [111]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omnw-1.4')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Error loading omnw-1.4: Package 'omnw-1.4' not found in
[nltk_data]     index


In [112]:
data.drop('length',axis=1,inplace=True)

In [113]:
data.head()

,sentimental,news
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...


In [114]:
import re
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words=text.split()
    words=[lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return "   ".join(words)

data['cleaned_news']=data['news'].apply(preprocess_text)
data

,sentimental,news,cleaned_news
0,neutral,"According to Gran , the company has no plans t...",according gran company plan move pro...
1,neutral,Technopolis plans to develop in stages an area...,technopolis plan develop stage area ...
2,negative,The international electronic industry company ...,international electronic industry compan...
3,positive,With the new production plant the company woul...,new production plant company would i...
4,positive,According to the company 's updated strategy f...,according company updated strategy yea...
...,...,...,...
4841,negative,LONDON MarketWatch -- Share prices ended lower...,london marketwatch share price ended ...
4842,neutral,Rinkuskiai 's beer sales fell by 6.5 per cent ...,rinkuskiai beer sale fell per cent ...
4843,negative,Operating profit fell to EUR 35.4 mn from EUR ...,operating profit fell eur mn eur m...
4844,negative,Net sales of the Paper segment decreased to EU...,net sale paper segment decreased eur...


In [115]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
data['label']=label_encoder.fit_transform(data['sentimental'])
data.head()

,sentimental,news,cleaned_news,label
0,neutral,"According to Gran , the company has no plans t...",according gran company plan move pro...,1
1,neutral,Technopolis plans to develop in stages an area...,technopolis plan develop stage area ...,1
2,negative,The international electronic industry company ...,international electronic industry compan...,0
3,positive,With the new production plant the company woul...,new production plant company would i...,2
4,positive,According to the company 's updated strategy f...,according company updated strategy yea...,2


In [116]:
from sklearn.model_selection import train_test_split
X=data['cleaned_news']
y=data['label']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [117]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=10000,ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [118]:
X_train_tfidf.shape

(3872, 10000)

In [119]:
X_train_tfidf.shape
print(X_train_tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 56365 stored elements and shape (3872, 10000)>
  Coords	Values
  (0, 7091)	0.14600958975920802
  (0, 8163)	0.18099339782653126
  (0, 8694)	0.6041813306026458
  (0, 6228)	0.17036864707775407
  (0, 4592)	0.21603909667467797
  (0, 6639)	0.28334840240688997
  (0, 9941)	0.1427129815803531
  (0, 7713)	0.1338837538299904
  (0, 7097)	0.25687942668498914
  (0, 8695)	0.31312566949137755
  (0, 6230)	0.21099324400789862
  (0, 4615)	0.28787132276684796
  (0, 6660)	0.23848380367996394
  (0, 7719)	0.19573878905661057
  (1, 8278)	0.24186788113825505
  (1, 7902)	0.31528228671127734
  (1, 5062)	0.26575010542433597
  (1, 3553)	0.2731658740970101
  (1, 5526)	0.2947191212550641
  (1, 6415)	0.18971894943505374
  (1, 9808)	0.30926102935755134
  (1, 7903)	0.3927079540195077
  (1, 5065)	0.3927079540195077
  (1, 3561)	0.41221377423347244
  (2, 9563)	0.529046585883605
  :	:
  (3871, 4613)	0.1258868118733716
  (3871, 1375)	0.19760396776174396
  (3871, 

In [120]:
print(X_train_tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 56365 stored elements and shape (3872, 10000)>
  Coords	Values
  (0, 7091)	0.14600958975920802
  (0, 8163)	0.18099339782653126
  (0, 8694)	0.6041813306026458
  (0, 6228)	0.17036864707775407
  (0, 4592)	0.21603909667467797
  (0, 6639)	0.28334840240688997
  (0, 9941)	0.1427129815803531
  (0, 7713)	0.1338837538299904
  (0, 7097)	0.25687942668498914
  (0, 8695)	0.31312566949137755
  (0, 6230)	0.21099324400789862
  (0, 4615)	0.28787132276684796
  (0, 6660)	0.23848380367996394
  (0, 7719)	0.19573878905661057
  (1, 8278)	0.24186788113825505
  (1, 7902)	0.31528228671127734
  (1, 5062)	0.26575010542433597
  (1, 3553)	0.2731658740970101
  (1, 5526)	0.2947191212550641
  (1, 6415)	0.18971894943505374
  (1, 9808)	0.30926102935755134
  (1, 7903)	0.3927079540195077
  (1, 5065)	0.3927079540195077
  (1, 3561)	0.41221377423347244
  (2, 9563)	0.529046585883605
  :	:
  (3871, 4613)	0.1258868118733716
  (3871, 1375)	0.19760396776174396
  (3871, 

In [121]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, classification_report

models = {
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier()
}

results = {}
trainded_models = {}

for name, model in models.items():
    
    # Train
    model.fit(X_train_tfidf, y_train)
    
    trainded_models[name] = model
    
    # Predict
    y_pred = model.predict(X_test_tfidf)
    
    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)
    
    results[name] = accuracy
    
    print(f"\n{name}")
    print("-" * 40)
    print("Accuracy:", accuracy)
    print(classification_report(y_test, y_pred))


Logistic Regression
----------------------------------------
Accuracy: 0.746900826446281
              precision    recall  f1-score   support

           0       0.80      0.41      0.54       118
           1       0.73      0.96      0.83       563
           2       0.79      0.46      0.58       287

    accuracy                           0.75       968
   macro avg       0.77      0.61      0.65       968
weighted avg       0.76      0.75      0.72       968


Naive Bayes
----------------------------------------
Accuracy: 0.6911157024793388
              precision    recall  f1-score   support

           0       0.95      0.15      0.26       118
           1       0.68      0.98      0.80       563
           2       0.70      0.35      0.47       287

    accuracy                           0.69       968
   macro avg       0.78      0.49      0.51       968
weighted avg       0.72      0.69      0.64       968


Linear SVM
----------------------------------------
Accuracy: 0.

In [122]:
for model, score in results.items():
    print(f"{model}: {score:.4f}")

Logistic Regression: 0.7469
Naive Bayes: 0.6911
Linear SVM: 0.7614
Random Forest: 0.7459
Decision Tree: 0.7014
KNN: 0.6591


In [123]:
import joblib
joblib.dump(trainded_models['Linear SVM'], 'svm_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

In [124]:
def predict_sentiment(text):
    cleaned_text = preprocess_text(text)
    text_tfidf = vectorizer.transform([cleaned_text])
    prediction = trainded_models['Linear SVM'].predict(text_tfidf)
    
    label_map={0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    return label_map[prediction[0]]

In [125]:
predict_sentiment("Tesla stock rises after record profits")

'Positive'

In [126]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
param_grid = {
    'C': [0.01, 0.5, 1,1.5, 2],
}
grid_search = GridSearchCV(LinearSVC(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_tfidf, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LinearSVC()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate parameter indexes are 

In [127]:
print(grid_search.best_params_)
print(grid_search.best_score_)

{'C': 1.5}
0.7391557889472369


In [128]:
best_svm = grid_search.best_estimator_

In [129]:
y_pred_best = best_svm.predict(X_test_tfidf)

In [130]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_best)

print(accuracy)

0.7572314049586777
